# 🌍 Example 5: Model Colouring Techniques

Welcome! In this tutorial, we will explore the different ways to color your 3D globes using `globe3d`'s modular `Colourer` subclasses. To ensure high-quality color rendering, each example generates a high-resolution globe with **1,000,000 nodes** and visualizes it using the built-in `model.preview()` helper.

## Setup & Imports

First, we import the required classes and scientific modules.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from globe3d import (
    GlobeModel,
    ConstantColourer,
    ImageColourer,
    GridColourer,
    PolygonColourer,
    PointColourer,
    GeographicGrid
)

## Example 1: Constant Colouring of a Simple Globe

Paints the entire globe a single solid RGB color using `ConstantColourer`.

In [ ]:
# 1. Initialize a globe
model = GlobeModel(n_points=20000, radius=40.0)

# 2. Apply a constant sky blue color
sky_blue = ConstantColourer([0.1, 0.5, 0.8])
model.outer.colour(sky_blue)

# 3. Export (commented out in place)
# model.export("../outputs/example_5_1_constant.obj")

# 4. Preview in 3D
model.preview()

## Example 2: Colour with Equirectangular Image

Generates a gradient texture image on the fly and samples colors from it using `ImageColourer`.

In [ ]:
# 1. Generate an equirectangular gradient texture
width, height = 360, 180
image = np.zeros((height, width, 3), dtype=np.float32)
for x in range(width):
    image[:, x, 0] = x / width        # Red gradient
    image[:, x, 1] = 1.0 - x / width  # Green gradient
for y in range(height):
    image[y, :, 2] = y / height       # Blue gradient

os.makedirs('../outputs', exist_ok=True)
plt.imsave('../outputs/demo_texture.png', image)

# 2. Initialize the globe
model = GlobeModel(n_points=20000, radius=40.0)

# 3. Apply color sampling from the image
model.outer.colour(ImageColourer('../outputs/demo_texture.png'))

# 4. Export (commented out in place)
# model.export("../outputs/example_5_2_image.obj")

# 5. Preview in 3D
model.preview()

## Example 3: Colour with a Grid

Loads a scientific topography dataset and maps heights to colors using Matplotlib's `'terrain'` colormap.

In [ ]:
# 1. Load scientific grid or fall back to synthetic topography
nc_path = "../inputs/ETOPO_2022_v1_60s_N90W180_surface.nc"
try:
    grid_data = GeographicGrid.from_netcdf(nc_path, lat_var='lat', lon_var='lon', data_var='z')
    grid_ds = GeographicGrid(
        lats=grid_data.lats[::10],
        lons=grid_data.lons[::10],
        grid=grid_data.grid[::10, ::10]
    )
except FileNotFoundError:
    print("ETOPO grid file not found. Generating synthetic topography data.")
    lats = np.linspace(-90, 90, 180)
    lons = np.linspace(-180, 180, 360)
    grid = np.zeros((180, 360))
    for i, lat in enumerate(lats):
        for j, lon in enumerate(lons):
            grid[i, j] = 5000.0 * np.sin(np.radians(lat)) * np.cos(np.radians(lon))
    grid_ds = GeographicGrid(lats, lons, grid)

# 2. Initialize the globe at High res
model = GlobeModel(n_points=1000000, radius=40.0)

# 3. Map heights to the 'terrain' colormap
model.outer.colour(GridColourer(grid_ds, colormap='terrain', vmin=-5000, vmax=5000))

# 4. Export (commented out in place)
# model.export("../outputs/example_5_3_grid.obj")

# 5. Preview in 3D
model.preview()

## Example 4: Colour with Shapefiles / Arrays

Floods land polygons green and oceans blue using `PolygonColourer` (falling back to array coordinates if the shapefile is missing), and paints red star markers at the coordinates of the 5 most populous capital cities using `PointColourer`.

In [ ]:
land_shp = "../inputs/coastlines/ne_110m_land.shp"
if os.path.exists(land_shp):
    land_data = land_shp
else:
    print("Natural Earth land shapefile not found. Using array coordinates for landmasses.")
    land_data = [
        # Americas
        np.array([[-120, -40], [-40, -40], [-40, 60], [-120, 60], [-120, -40]]),
        # Eurasia + Africa
        np.array([[-20, -30], [140, -30], [140, 70], [-20, 70], [-20, -30]]),
        # Australia
        np.array([[110, -40], [150, -40], [150, -10], [110, -10], [110, -40]])
    ]

# 1. Initialize the globe
model = GlobeModel(n_points=1000000, radius=40.0)

# 2. Flood land green and ocean blue
model.outer.colour(PolygonColourer(land_data, color=[0.1, 0.7, 0.2], background_color=[0.1, 0.3, 0.8], flood_inside=True))

# 3. Add red stars at the coordinates of the 5 most populous capitals
capitals = np.array([
    [139.6917, 35.6895],  # Tokyo
    [77.2090, 28.6139],    # Delhi
    [116.4074, 39.9042],   # Beijing
    [90.4125, 23.8103],    # Dhaka
    [31.2357, 30.0444]     # Cairo
])
model.outer.colour(PointColourer(capitals, color=[1.0, 0.0, 0.0], radius_degrees=2.5, marker_shape="star"))

# 4. Export (commented out in place)
# model.export("../outputs/example_5_4_shapes.obj")

# 5. Preview in 3D
model.preview()

## Example 5: Controlling Which Area to Colour

Generates a high-resolution hollow globe and demonstrates how to apply separate colors to the outer surface, inner cavity walls, and inner shell using vertex subsets.

In [ ]:
# 1. Generate hollow model (1M outer points, 500k inner points)
model = GlobeModel(n_points=20000, radius=40.0, hollow=True, inner_ratio=0.8, inner_n_points=500000)

# 2. Generate a simple grid for topography
lats = np.linspace(-90, 90, 180)
lons = np.linspace(-180, 180, 360)
grid = np.zeros((180, 360))
for i, lat in enumerate(lats):
    for j, lon in enumerate(lons):
        grid[i, j] = 5000.0 * np.sin(np.radians(lat)) * np.cos(np.radians(lon))
grid_ds = GeographicGrid(lats, lons, grid)

# 3. Color outer shell faces with terrain topography
model.outer.colour(GridColourer(grid_ds, colormap='terrain', vmin=-5000, vmax=5000))

# 4. Color the inner shell surface red
model.inner.colour(ConstantColourer([1.0, 0.2, 0.2]))

# 5. Export hemispheres (commented out in place)
# model.export_hemispheres("../outputs/example_5_5_top.obj", "../outputs/example_5_5_bottom.obj", engine='manifold')

# 6. Preview in 3D
model.preview('upper')